# V6.3-ROI-A: Controlled Canonical Defect-ROI Experiment

This notebook is the **next controlled experiment** after the V6.2E localization audit.

The audit showed strong evidence that three defect classes are often centered correctly but
their bbox extent is unstable at higher IoU:

- `insulator_broken`: actual pass@0.75 ≈ 0.540 vs center-only ≈ 0.766
- `insulator_flashover`: actual ≈ 0.392 vs center-only ≈ 0.665
- `broken_strand`: actual ≈ 0.453 vs center-only ≈ 0.729

The 20-px / 1.2× what-if sweep also produced large pass@0.75 gains, but that sweep was only
a diagnostic upper bound, **not mAP**.

So this notebook does **not** overwrite V6.2E. It creates one conservative candidate:

```text
V6.3-ROI-A
insulator            unchanged
bird_nest            unchanged
foreign_object       unchanged

insulator_broken     1.20× context, minimum side 20 px @ 1024 reference scale
insulator_flashover  1.20× context, minimum side 20 px @ 1024 reference scale
broken_strand        1.20× context, no minimum-side expansion
```

The bbox center is preserved exactly. Expansion is capped symmetrically at the image boundary,
so edge boxes do not drift.

Then it trains **YOLO11n @640 only**, using the exact V6.2E baseline recipe. This isolates the
effect of the annotation target. TEST is not restored, transformed, predicted, or evaluated here.

Promotion gate:

```text
VAL mAP50 >= 0.80
VAL (mAP50 - mAP50:95) <= 0.30
```

If ROI-A fails, do not jump to RT-DETR and do not silently increase boxes to 1.4×. Diagnose first.


In [ ]:
from pathlib import Path
import os, sys, json, csv, shutil, subprocess, hashlib, zipfile, math, re, time, threading
from collections import Counter, defaultdict

W = Path("/kaggle/working")
TMP = Path("/tmp/v6p3_roi_a_clean")
BASE_EXTRACT = TMP/"base"
OVERLAY_EXTRACT = TMP/"overlay"
RECON_LABELS = TMP/"reconciled_labels"

OUT = W/"v6p3_roi_a"
REPORT = OUT/"report"
PREVIEW = OUT/"preview"
RUNTIME = TMP/"runtime_train_val"
RUNS = W/"v6p3_roi_a_runs"

for p in [TMP, BASE_EXTRACT, OVERLAY_EXTRACT, RECON_LABELS, OUT, REPORT, PREVIEW, RUNS]:
    p.mkdir(parents=True, exist_ok=True)

FINAL_TAG = "v6p2e_23562a45b343"
PARENT_TAG = "v6p1_5d0b1149891c"
BASE_TAG = "56b7ea194ee5"
OVERLAY_TAG = "ee2d3c4497fd"

DATA_ROOT = "stash://bunpmc/projects/yolov11sdi/datasets"
FINAL_ROOT = f"{DATA_ROOT}/v6p2e/{FINAL_TAG}"
PARENT_ROOT = f"{DATA_ROOT}/v6p1/{PARENT_TAG}"

BASE_STASH = f"{DATA_ROOT}/curated_v6_rs1280_{BASE_TAG}.tar"
OVERLAY_STASH = f"{PARENT_ROOT}/v6p1_fotl_overlay_{OVERLAY_TAG}.tar.gz"
FREEZE_STASH = f"{FINAL_ROOT}/v6p2e.freeze.json"
MANIFEST_STASH = f"{FINAL_ROOT}/v6p2e_split_manifest.csv"
RECON_STASH = f"{FINAL_ROOT}/v6p2e_reconciled_labels.zip"

NAMES = [
    "insulator",
    "insulator_broken",
    "insulator_flashover",
    "bird_nest",
    "foreign_object",
    "broken_strand",
]

EXPECTED_SPLITS = {"train":17483, "val":2272, "test":2012}
EXPECTED_TRAIN_BOXES = [6226,6236,1249,6085,667,2002]
EXPECTED_VAL_BOXES = [778,791,163,821,98,252]

IMG_EXTS={".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"}

# ----- Locked candidate policy -----
REFERENCE_IMGSZ = 1024

ROI_POLICY = {
    1: {"name":"insulator_broken",    "expand":1.20, "min_side_input_px":20},
    2: {"name":"insulator_flashover", "expand":1.20, "min_side_input_px":20},
    5: {"name":"broken_strand",       "expand":1.20, "min_side_input_px":0},
}

POLICY_PAYLOAD = {
    "policy_name":"V6.3-ROI-A",
    "parent_dataset_tag":FINAL_TAG,
    "reference_imgsz":REFERENCE_IMGSZ,
    "classes":ROI_POLICY,
    "center_preserving":True,
    "boundary_behavior":"symmetrically cap width/height so center never moves",
    "test_policy":"TEST excluded from this development experiment",
}

POLICY_HASH = hashlib.sha256(
    json.dumps(POLICY_PAYLOAD,sort_keys=True).encode()
).hexdigest()[:12]

CANDIDATE_TAG = f"v6p3_roi_a_{POLICY_HASH}"

REMOTE_DATA = (
    "stash://bunpmc/projects/yolov11sdi/datasets/"
    f"v6p3_roi_a/{CANDIDATE_TAG}"
)
REMOTE_CKPT = (
    "stash://bunpmc/projects/yolov11sdi/checkpoints/"
    f"v6p3_roi_a/{CANDIDATE_TAG}"
)
REMOTE_REPORT = (
    "stash://bunpmc/projects/yolov11sdi/reports/"
    f"v6p3_roi_a/{CANDIDATE_TAG}"
)

# Exact baseline-control training recipe.
SEED = 42
WORKERS = 4
UPLOAD_EVERY_MIN = 15

TRAIN_YOLO11N = True
RUN_TEST = False  # hard false by design in this notebook

BASELINE_V6P2E_YOLO11N = {
    "mAP50": 0.768,
    "mAP50_95": 0.413,
    "gap": 0.355,
}

ACCEPT_MAP50 = 0.80
ACCEPT_MAX_GAP = 0.30

print("Candidate tag:", CANDIDATE_TAG)
print(json.dumps(POLICY_PAYLOAD,indent=2))


## 1. Runtime + Camber

In [ ]:
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "ultralytics", "pandas", "pyyaml", "pillow"
])

import numpy as np
import pandas as pd
import yaml
import torch
from PIL import Image, ImageDraw, ImageOps
from ultralytics import YOLO
from kaggle_secrets import UserSecretsClient

print("Torch:",torch.__version__)
print("CUDA:",torch.cuda.is_available())
print("GPU count:",torch.cuda.device_count())
print("GPUs:",[
    torch.cuda.get_device_name(i)
    for i in range(torch.cuda.device_count())
])

assert torch.cuda.device_count() >= 2, (
    "Use Kaggle T4 x2 so this controlled run matches the V6.2E baseline setup."
)

key=UserSecretsClient().get_secret("CAMBER_API_KEY")
if not key:
    raise RuntimeError("Missing CAMBER_API_KEY")

os.environ["CAMBER_API_KEY"]=key

if shutil.which("camber") is None and not (Path.home()/".camber/bin/camber").exists():
    subprocess.check_call(
        "curl -sL https://cli.cambercloud.com/install-v2.sh | bash",
        shell=True
    )

os.environ["PATH"]=(
    f"{Path.home()}/.camber/bin:"
    f"{Path.home()}/.local/bin:"
    + os.environ.get("PATH","")
)

CAMBER=shutil.which("camber") or str(Path.home()/".camber/bin/camber")

def stash_cp(src,dst,required=True):
    dst=Path(dst)
    dst.parent.mkdir(parents=True,exist_ok=True)
    r=subprocess.run(
        [CAMBER,"stash","cp",str(src),str(dst)],
        text=True,capture_output=True,env=os.environ.copy()
    )
    ok=r.returncode==0 and dst.exists()
    if required and not ok:
        raise RuntimeError(
            f"Camber copy failed: {src}\n"
            f"{r.stdout[-1000:]}\n{r.stderr[-1600:]}"
        )
    return ok

def stash_put(src,dst,required=True):
    src=Path(src)
    r=subprocess.run(
        [CAMBER,"stash","cp",str(src),str(dst)],
        text=True,capture_output=True,env=os.environ.copy()
    )
    ok=r.returncode==0
    if required and not ok:
        raise RuntimeError(
            f"Camber upload failed: {dst}\n"
            f"{r.stdout[-1000:]}\n{r.stderr[-1600:]}"
        )
    if ok:
        print("Uploaded:",dst)
    return ok


## 2. Clean restore accepted V6.2E lineage

Only TRAIN + VAL are reconstructed. TEST stays outside this development experiment.


In [ ]:
freeze_local = TMP/"v6p2e.freeze.json"
manifest_local = TMP/"v6p2e_split_manifest.csv"
recon_zip = TMP/"v6p2e_reconciled_labels.zip"
base_tar = TMP/f"base_{BASE_TAG}.tar"
overlay_tar = TMP/f"overlay_{OVERLAY_TAG}.tar.gz"

# IMPORTANT:
# Never reuse previously extracted /tmp source trees. Earlier V6.3 notebook
# versions used "if data.yaml exists -> reuse", which can preserve a stale or
# partial extraction across reruns in the same Kaggle session.
for p in [BASE_EXTRACT, OVERLAY_EXTRACT, RECON_LABELS]:
    if p.exists():
        shutil.rmtree(p)
    p.mkdir(parents=True, exist_ok=True)

for p in [freeze_local, manifest_local, recon_zip, base_tar, overlay_tar]:
    p.unlink(missing_ok=True)

print("Downloading accepted V6.2E metadata...")
stash_cp(FREEZE_STASH, freeze_local)
stash_cp(MANIFEST_STASH, manifest_local)
stash_cp(RECON_STASH, recon_zip)

freeze = json.loads(freeze_local.read_text())
manifest = pd.read_csv(manifest_local)

assert freeze["final_tag"] == FINAL_TAG
assert freeze["freeze_ready"] is True
assert len(manifest) == sum(freeze["split_counts"].values()), (
    len(manifest), freeze["split_counts"]
)

print("Downloading immutable source archives from Camber...")
stash_cp(BASE_STASH, base_tar)
stash_cp(OVERLAY_STASH, overlay_tar)

# Archive sanity before extraction.
def tar_names(path, gz=False):
    flag = "-tzf" if gz else "-tf"
    r = subprocess.run(
        ["tar", flag, str(path)],
        text=True, capture_output=True
    )
    if r.returncode != 0:
        raise RuntimeError(
            f"Cannot list archive {path}\n{r.stderr[-2000:]}"
        )
    return r.stdout.splitlines()

base_names = tar_names(base_tar, gz=False)
overlay_names = tar_names(overlay_tar, gz=True)

print("BASE archive entries:", len(base_names))
print("OVERLAY archive entries:", len(overlay_names))

# This exact file is the one that exposed stale/mismatched extraction earlier.
# Do not require the basename to exist because a previous curation stage may
# legitimately rename files; this is only a diagnostic.
_known_problem_basename = "fotl_image_val159_6530eaae.jpg"
print(
    "Known-problem basename present in raw OVERLAY archive:",
    any(Path(x).name == _known_problem_basename for x in overlay_names)
)

print("Extracting clean source trees...")
subprocess.check_call(["tar","-xf",str(base_tar),"-C",str(BASE_EXTRACT)])
subprocess.check_call(["tar","-xzf",str(overlay_tar),"-C",str(OVERLAY_EXTRACT)])

# Reconciled labels are also rebuilt cleanly.
with zipfile.ZipFile(recon_zip) as z:
    z.extractall(RECON_LABELS)

def norm_names(names):
    if isinstance(names, dict):
        return [names[k] for k in sorted(names, key=lambda x:int(x))]
    return list(names)

def find_root(root):
    matches=[]
    for yp in Path(root).rglob("data.yaml"):
        try:
            cfg = yaml.safe_load(yp.read_text())
            if norm_names(cfg.get("names",[])) == NAMES:
                matches.append(yp.parent)
        except Exception:
            pass
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one six-class dataset root under {root}; "
            f"found {len(matches)}: {matches}"
        )
    return matches[0]

BASE = find_root(BASE_EXTRACT)
OVERLAY = find_root(OVERLAY_EXTRACT)

print("Freeze:", FINAL_TAG)
print("Manifest rows:", len(manifest))
print("BASE:", BASE)
print("OVERLAY:", OVERLAY)
print("Manifest columns:", list(manifest.columns))
manifest_dev = manifest[manifest["split"].isin(["train","val"])].copy()
print("TRAIN+VAL manifest rows:", len(manifest_dev))
assert len(manifest_dev) == EXPECTED_SPLITS["train"] + EXPECTED_SPLITS["val"]


# Do not retain archive files after a successful extraction.
base_tar.unlink(missing_ok=True)
overlay_tar.unlink(missing_ok=True)
recon_zip.unlink(missing_ok=True)


## 3. Resolve exact TRAIN/VAL source bytes

In [ ]:
PROVENANCE_COL_CANDIDATES = [
    "source_dataset", "original_source", "dataset", "dataset_source",
    "raw_source", "provenance", "source_name",
]

IMG_EXTS={".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"}

def sha256_file(path, chunk=4*1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(chunk),b""):
            h.update(b)
    return h.hexdigest()

def explicit_provenance(row):
    for c in PROVENANCE_COL_CANDIDATES:
        if c in manifest.columns:
            v = str(row.get(c,"")).strip()
            if v and v.lower() not in {"nan","none",""}:
                return v, f"manifest:{c}"
    return None, None

def parse_old_path(row):
    old_path = str(row["image_path_source"]).replace("\\","/")
    old_split = str(row["previous_v6p1_split"])

    # Prefer the declared split marker, but tolerate rows whose logical split lineage
    # no longer matches the physical archive split.
    marker = f"/images/{old_split}/"
    if marker in old_path:
        suffix = old_path.split(marker,1)[1]
    else:
        m = re.search(r"/images/(train|val|test)/(.*)$", old_path)
        if not m:
            raise RuntimeError(f"Cannot parse source path: {old_path}")
        suffix = m.group(2)

    if suffix.startswith("base/"):
        family = "base"
        rel = suffix[len("base/"):]
        preferred_root = BASE
    elif suffix.startswith("fotl/"):
        family = "fotl"
        rel = suffix[len("fotl/"):]
        preferred_root = OVERLAY
    else:
        raise RuntimeError(f"Unknown source suffix: {suffix}")

    parts = Path(rel).parts
    path_group = parts[0] if len(parts) > 1 else family
    return old_path, old_split, family, rel, preferred_root, path_group

# Build cheap basename indexes once for BOTH immutable source artifacts.
ROOT_NAME = {
    str(BASE): "base",
    str(OVERLAY): "fotl",
}
ALL_ROOTS=[BASE,OVERLAY]

BASENAME_INDEX=defaultdict(list)
for rr in ALL_ROOTS:
    img_root=Path(rr)/"images"
    if not img_root.exists():
        continue
    for p in img_root.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            BASENAME_INDEX[p.name].append((Path(rr),p))

print("Indexed physical source images by basename:",
      sum(len(v) for v in BASENAME_INDEX.values()))

_SHA_INDEX=None

def build_sha_index():
    """
    Expensive fallback, built at most once.
    It scans both BASE and OVERLAY and maps SHA256 -> actual image path/root.
    This handles artifacts where a manifest's original filename/path is no longer
    present but the exact image bytes survived under a renamed path.
    """
    global _SHA_INDEX
    if _SHA_INDEX is not None:
        return _SHA_INDEX

    idx=defaultdict(list)
    n=0
    for rr in ALL_ROOTS:
        img_root=Path(rr)/"images"
        if not img_root.exists():
            continue
        for p in img_root.rglob("*"):
            if not (p.is_file() and p.suffix.lower() in IMG_EXTS):
                continue
            idx[sha256_file(p)].append((Path(rr),p))
            n+=1
            if n%5000==0:
                print("SHA-indexed",n,"source images...")

    _SHA_INDEX=idx
    print("✅ SHA index built:",n,"images,",len(idx),"unique hashes")
    return _SHA_INDEX

def label_for_actual_image(actual_root, ip, rel_hint=None):
    actual_root=Path(actual_root)
    ip=Path(ip)

    try:
        rel_from_images=ip.relative_to(actual_root/"images")
        lp=(actual_root/"labels"/rel_from_images).with_suffix(".txt")
        if lp.exists():
            return lp, str(rel_from_images.parts[0])
    except Exception:
        rel_from_images=None

    # Fallback by basename.
    candidates=list((actual_root/"labels").rglob(ip.stem+".txt"))
    if rel_hint is not None:
        hint_name=Path(rel_hint).with_suffix(".txt").name
        hinted=[p for p in candidates if p.name==hint_name]
        if len(hinted)==1:
            candidates=hinted

    if len(candidates)==1:
        lp=candidates[0]
        try:
            physical_split=str(lp.relative_to(actual_root/"labels").parts[0])
        except Exception:
            physical_split="unknown"
        return lp, physical_split

    # We still return a derived path if possible so caller can emit a precise error.
    if rel_from_images is not None:
        return (actual_root/"labels"/rel_from_images).with_suffix(".txt"), str(rel_from_images.parts[0])

    return actual_root/"labels"/(ip.stem+".txt"), "unknown"

def resolve_physical_pair(preferred_root, old_split, rel, expected_sha):
    """
    Resolve exact source bytes with progressively broader search:

      A. preferred root + declared split
      B. preferred root + any split
      C. same basename across BOTH BASE and OVERLAY
      D. exact SHA256 across BOTH BASE and OVERLAY, even if renamed

    A file is accepted only when SHA256 == manifest SHA256.
    Returns:
      actual_root, image_path, label_path, physical_source_split, resolution_mode
    """
    preferred_root=Path(preferred_root)
    rel=Path(rel)

    candidates=[]
    seen=set()

    def add(actual_root,p,mode):
        actual_root=Path(actual_root)
        p=Path(p)
        key=(str(actual_root),str(p))
        if key not in seen:
            seen.add(key)
            candidates.append((actual_root,p,mode))

    # A: exact manifest lineage path.
    add(preferred_root, preferred_root/"images"/old_split/rel, "preferred_declared_split")

    # B: same relative path in any split of preferred root.
    for sp in ["train","val","test"]:
        add(preferred_root, preferred_root/"images"/sp/rel, "preferred_other_split")

    # C: basename across both artifacts.
    for actual_root,p in BASENAME_INDEX.get(rel.name,[]):
        add(actual_root,p,"cross_root_same_basename")

    existing=[(rr,p,mode) for rr,p,mode in candidates if p.exists() and p.is_file()]

    for actual_root,ip,mode in existing:
        if sha256_file(ip)==expected_sha:
            lp,physical_split=label_for_actual_image(actual_root,ip,rel)
            return actual_root,ip,lp,physical_split,mode

    # D: Exact-byte fallback across both artifacts.
    sha_index=build_sha_index()
    matches=sha_index.get(expected_sha,[])
    if matches:
        # Prefer the manifest-indicated artifact if the hash exists in both.
        matches=sorted(matches,key=lambda x: 0 if Path(x[0])==preferred_root else 1)
        actual_root,ip=matches[0]
        lp,physical_split=label_for_actual_image(actual_root,ip,rel)
        return actual_root,ip,lp,physical_split,"cross_root_sha256"

    # At this point we have freshly downloaded + freshly extracted both immutable
    # source archives in this same run, and searched them by exact SHA256.
    # Therefore this is a genuine artifact/manifest mismatch, not a path-cache issue.
    missing_row = {
        "expected_sha": expected_sha,
        "preferred_root": str(preferred_root),
        "old_split": old_split,
        "relative_path": str(rel),
        "same_basename_candidates": [
            str(p) for _,p in BASENAME_INDEX.get(rel.name,[])[:20]
        ],
    }
    diag = OUT/"MISSING_SOURCE_BYTES.json"
    diag.write_text(json.dumps(missing_row, indent=2))
    raise FileNotFoundError(
        "CLEAN-RESTORE INTEGRITY FAILURE.\n"
        "The exact manifest SHA256 is absent from BOTH freshly restored source artifacts.\n"
        f"Diagnostic written to: {diag}\n"
        + json.dumps(missing_row, indent=2)
    )

rows=[]

for idx,row in manifest_dev.iterrows():
    old_path,old_split,family,rel,preferred_root,path_group=parse_old_path(row)
    expected_sha=str(row["sha256"])

    prov,prov_basis=explicit_provenance(row)
    if prov is None:
        if path_group not in {"base","fotl"} and len(Path(rel).parts)>1:
            prov=path_group
            prov_basis="path_subdir"
        else:
            prov=family
            prov_basis="coarse_family_only"

    actual_root,ip,original_lp,physical_source_split,resolution_mode=resolve_physical_pair(
        preferred_root=preferred_root,
        old_split=old_split,
        rel=rel,
        expected_sha=expected_sha,
    )

    # If the manifest says this sample used a reconciled label, that label wins.
    rp=RECON_LABELS/f"{expected_sha}.txt"
    if str(row.get("resolution","")).startswith("merged"):
        if not rp.exists():
            raise FileNotFoundError(f"Missing reconciled label: {rp}")
        lp=rp
    else:
        lp=original_lp
        if not lp.exists():
            raise FileNotFoundError(
                "Resolved exact image bytes but corresponding label is missing.\n"
                f"actual_root={actual_root}\n"
                f"image={ip}\n"
                f"derived_label={lp}\n"
                f"sha={expected_sha}\n"
                f"resolution_mode={resolution_mode}"
            )

    rows.append({
        "row_id":idx,
        "sha256":expected_sha,
        "split":str(row["split"]),
        "previous_split":old_split,
        "physical_source_split":physical_source_split,
        "coarse_family":family,
        "actual_artifact":ROOT_NAME.get(str(actual_root),str(actual_root)),
        "provenance":str(prov),
        "provenance_basis":prov_basis,
        "relative_path":rel,
        "image_path":str(ip),
        "label_path":str(lp),
        "path_resolution_mode":resolution_mode,
        "resolution":str(row.get("resolution","")),
    })

rec=pd.DataFrame(rows)

if (rec.provenance_basis=="coarse_family_only").mean()>0.80:
    print("⚠️ Most original dataset provenance is no longer recoverable from this artifact.")
    print("   ROI-A can still run, but source-by-source conclusions will be limited.")

print("\nResolution modes:")
display(rec["path_resolution_mode"].value_counts().rename_axis("mode").reset_index(name="images"))

cross_artifact=rec[
    ((rec.coarse_family=="fotl") & (rec.actual_artifact!="fotl")) |
    ((rec.coarse_family=="base") & (rec.actual_artifact!="base"))
]
print("Cross-artifact recoveries:",len(cross_artifact))
if len(cross_artifact):
    display(cross_artifact[
        ["coarse_family","actual_artifact","previous_split",
         "physical_source_split","relative_path","path_resolution_mode"]
    ].head(50))

split_mismatch=rec[rec["previous_split"]!=rec["physical_source_split"]]
print("Physical split relocations resolved:",len(split_mismatch))
if len(split_mismatch):
    display(split_mismatch[
        ["coarse_family","actual_artifact","previous_split",
         "physical_source_split","relative_path","path_resolution_mode"]
    ].head(50))

display(
    rec.groupby(["provenance","provenance_basis","split"])
       .size().reset_index(name="images")
       .sort_values("images",ascending=False)
       .head(100)
)

rec.to_csv(OUT/"recovered_provenance.csv",index=False)


## 4. Parse original TRAIN/VAL labels

In [ ]:
def parse_label(path):
    boxes = []
    for line_no,line in enumerate(Path(path).read_text(errors="ignore").splitlines(),1):
        if not line.strip():
            continue
        p = line.split()
        if len(p) < 5:
            raise RuntimeError(f"Malformed label: {path}:{line_no}")
        cid = int(float(p[0]))
        x,y,w,h = map(float,p[1:5])
        boxes.append({
            "line_no":line_no,
            "class_id":cid,
            "class":NAMES[cid],
            "cx":x,"cy":y,"w":w,"h":h,
        })
    return boxes

box_rows = []
image_rows = []

for _,r in rec.iterrows():
    boxes = parse_label(r.label_path)
    image_rows.append({**r.to_dict(), "n_boxes":len(boxes)})

    with Image.open(r.image_path) as im:
        Wd,Hd = im.size

    for b in boxes:
        area = b["w"]*b["h"]
        ar = max(b["w"]/b["h"], b["h"]/b["w"]) if b["w"]>0 and b["h"]>0 else np.inf
        box_rows.append({
            **r.to_dict(),
            **b,
            "area":area,
            "aspect_ratio":ar,
            "pixel_w":b["w"]*Wd,
            "pixel_h":b["h"]*Hd,
        })

images_df = pd.DataFrame(image_rows)
boxes_df = pd.DataFrame(box_rows)

print("Images:",len(images_df))
print("Boxes:",len(boxes_df))
display(boxes_df.groupby(["provenance","class"]).size().reset_index(name="boxes"))


In [ ]:
def count_boxes(df, split):
    g=(
        df[df["split"]==split]
        .groupby("class_id").size()
        .reindex(range(len(NAMES)),fill_value=0)
        .astype(int)
        .tolist()
    )
    return g

got_train=count_boxes(boxes_df,"train")
got_val=count_boxes(boxes_df,"val")

print("Original TRAIN boxes:",got_train)
print("Original VAL boxes  :",got_val)

assert got_train==EXPECTED_TRAIN_BOXES,(got_train,EXPECTED_TRAIN_BOXES)
assert got_val==EXPECTED_VAL_BOXES,(got_val,EXPECTED_VAL_BOXES)

assert len(images_df[images_df["split"]=="train"])==EXPECTED_SPLITS["train"]
assert len(images_df[images_df["split"]=="val"])==EXPECTED_SPLITS["val"]

print("✅ Original TRAIN/VAL distributions reproduced.")
print("✅ TEST has not been reconstructed into the development runtime.")


## 5. Apply center-preserving canonical ROI policy

For a modified class:

```text
target_w = max(original_w × 1.20, min_side_reference)
target_h = max(original_h × 1.20, min_side_reference)
```

The minimum side is defined relative to a canonical 1024 letterbox scale. Then width/height are
symmetrically capped by the nearest image boundary so the bbox center remains **exactly unchanged**.

No class IDs change. No boxes are added or removed.


Numerical note: V6.2E normalized labels are rounded. A boundary box may therefore
exceed `[0,1]` by around `1e-6` after decimal reconstruction. ROI-A permits only a
microscopic center-preserving shrink up to `5e-6` to bring such boxes back onto the
exact image boundary. Anything larger still hard-fails.


In [ ]:
def transform_box(cid,cx,cy,w,h,image_w,image_h):
    """
    YOLO normalized bbox -> center-preserving canonical ROI.
    Returns transformed bbox + diagnostics.
    """
    cid=int(cid)
    cx=float(cx); cy=float(cy); w=float(w); h=float(h)

    if cid not in ROI_POLICY:
        return {
            "cx":cx,"cy":cy,"w":w,"h":h,
            "modified":False,
            "boundary_limited_w":False,
            "boundary_limited_h":False,
            "min_rule_active":False,
            "rounding_shrink_w":0.0,
            "rounding_shrink_h":0.0,
        }

    cfg=ROI_POLICY[cid]
    expand=float(cfg["expand"])
    min_input=float(cfg["min_side_input_px"])

    # Original-image pixels per canonical input pixel under 1024 letterbox scaling.
    scale_ref=min(REFERENCE_IMGSZ/image_w,REFERENCE_IMGSZ/image_h)

    if min_input>0:
        min_side_orig_px=min_input/max(scale_ref,1e-12)
        min_w_norm=min_side_orig_px/image_w
        min_h_norm=min_side_orig_px/image_h
    else:
        min_w_norm=0.0
        min_h_norm=0.0

    requested_w=max(w*expand,min_w_norm)
    requested_h=max(h*expand,min_h_norm)

    # Keep center fixed. Largest symmetric box possible around this center:
    cap_w=max(0.0,2.0*min(cx,1.0-cx))
    cap_h=max(0.0,2.0*min(cy,1.0-cy))

    new_w=min(requested_w,cap_w)
    new_h=min(requested_h,cap_h)

    # V6.2E labels are stored with finite decimal precision. A box that was valid
    # in original pixel coordinates can therefore land microscopically outside [0,1]
    # after normalization, e.g. x2 = 1.0000005. In that case a center-preserving
    # boundary cap can make width/height smaller by ~1e-6. This is numerical
    # rounding, not a semantic bbox shrink.
    ROUNDING_TOL = 5e-6

    shrink_w = max(0.0, w - new_w)
    shrink_h = max(0.0, h - new_h)

    if shrink_w > ROUNDING_TOL or shrink_h > ROUNDING_TOL:
        raise RuntimeError(
            f"Center-preserving cap materially shrank bbox: "
            f"cid={cid}, orig={(cx,cy,w,h)}, new={(cx,cy,new_w,new_h)}, "
            f"shrink={(shrink_w,shrink_h)}"
        )

    return {
        "cx":cx,
        "cy":cy,
        "w":new_w,
        "h":new_h,
        "modified":(
            abs(new_w-w)>1e-12 or abs(new_h-h)>1e-12
        ),
        "boundary_limited_w":requested_w>cap_w+1e-12,
        "boundary_limited_h":requested_h>cap_h+1e-12,
        "min_rule_active":(
            min_input>0 and (min_w_norm>w*expand or min_h_norm>h*expand)
        ),
        "rounding_shrink_w":shrink_w,
        "rounding_shrink_h":shrink_h,
    }

transformed_rows=[]

for _,r in boxes_df.iterrows():
    t=transform_box(
        r.class_id,r.cx,r.cy,r.w,r.h,
        int(round(r.pixel_w/r.w)) if r.w>0 else 1,
        int(round(r.pixel_h/r.h)) if r.h>0 else 1,
    )

    transformed_rows.append({
        **r.to_dict(),
        "new_cx":t["cx"],
        "new_cy":t["cy"],
        "new_w":t["w"],
        "new_h":t["h"],
        "new_area":t["w"]*t["h"],
        "area_ratio":(t["w"]*t["h"])/(r.w*r.h+1e-12),
        "modified":t["modified"],
        "boundary_limited_w":t["boundary_limited_w"],
        "boundary_limited_h":t["boundary_limited_h"],
        "min_rule_active":t["min_rule_active"],
        "rounding_shrink_w":t["rounding_shrink_w"],
        "rounding_shrink_h":t["rounding_shrink_h"],
    })

tboxes=pd.DataFrame(transformed_rows)

# Hard invariants.
assert np.allclose(tboxes.cx,tboxes.new_cx,atol=1e-12)
assert np.allclose(tboxes.cy,tboxes.new_cy,atol=1e-12)
assert (tboxes.new_w>0).all() and (tboxes.new_h>0).all()
assert (tboxes.new_w<=1+1e-12).all() and (tboxes.new_h<=1+1e-12).all()
assert ((tboxes.new_cx-tboxes.new_w/2)>=-1e-9).all()
assert ((tboxes.new_cy-tboxes.new_h/2)>=-1e-9).all()
assert ((tboxes.new_cx+tboxes.new_w/2)<=1+1e-9).all()
assert ((tboxes.new_cy+tboxes.new_h/2)<=1+1e-9).all()

print("Modified boxes:",int(tboxes.modified.sum()),"/",len(tboxes))

rounding_adjusted = tboxes[
    (tboxes.rounding_shrink_w > 0) |
    (tboxes.rounding_shrink_h > 0)
]
print(
    "Microscopic boundary-rounding adjustments:",
    len(rounding_adjusted),
    "| max dw=",
    float(tboxes.rounding_shrink_w.max()),
    "| max dh=",
    float(tboxes.rounding_shrink_h.max()),
)

stats=[]
for cls,g in tboxes.groupby("class"):
    stats.append({
        "class":cls,
        "boxes":len(g),
        "modified_rate":float(g.modified.mean()),
        "median_area_ratio":float(g.area_ratio.median()),
        "p90_area_ratio":float(g.area_ratio.quantile(.90)),
        "boundary_limited_rate":float(
            (g.boundary_limited_w | g.boundary_limited_h).mean()
        ),
        "min_rule_active_rate":float(g.min_rule_active.mean()),
    })

roi_stats=pd.DataFrame(stats)
display(roi_stats)
roi_stats.to_csv(REPORT/"roi_transform_stats.csv",index=False)

tboxes.to_csv(
    REPORT/"roi_transform_manifest_train_val.csv.gz",
    index=False,
    compression="gzip"
)

(OUT/"roi_policy.json").write_text(
    json.dumps({
        **POLICY_PAYLOAD,
        "candidate_tag":CANDIDATE_TAG,
        "policy_hash":POLICY_HASH,
    },indent=2)
)

print("✅ Center-preserving ROI transform passed geometry gates.")


## 6. Visual QA sheets

Red = original V6.2E box  
Green = ROI-A box

The sheets intentionally oversample tiny/small modified boxes because those are where the 20-px
minimum rule matters most.


In [ ]:
rng=np.random.default_rng(SEED)

def draw_pair(img_path,orig,new,label):
    with Image.open(img_path) as im:
        im=ImageOps.exif_transpose(im).convert("RGB")
        Wd,Hd=im.size
        d=ImageDraw.Draw(im)

        def xyxy(cx,cy,w,h):
            return [
                (cx-w/2)*Wd,
                (cy-h/2)*Hd,
                (cx+w/2)*Wd,
                (cy+h/2)*Hd,
            ]

        ow=max(2,int(min(Wd,Hd)/350))

        d.rectangle(
            xyxy(orig["cx"],orig["cy"],orig["w"],orig["h"]),
            outline="red",width=ow
        )
        d.rectangle(
            xyxy(new["cx"],new["cy"],new["w"],new["h"]),
            outline="green",width=ow
        )

        txt=(
            f"{label}  area x{new['w']*new['h']/(orig['w']*orig['h']+1e-12):.2f}"
        )
        d.rectangle([0,0,min(Wd,760),30],fill="black")
        d.text((6,7),txt,fill="white")

        im.thumbnail((420,320))
        canvas=Image.new("RGB",(420,320))
        canvas.paste(im,((420-im.width)//2,(320-im.height)//2))
        return canvas

preview_paths=[]

for cid,cfg in ROI_POLICY.items():
    cls=NAMES[cid]
    g=tboxes[(tboxes.class_id==cid)&(tboxes.modified)].copy()

    if len(g)==0:
        continue

    # Half from smallest-area examples, half random.
    n_total=min(48,len(g))
    n_small=n_total//2
    small=g.nsmallest(n_small,"area")
    remain=g.drop(index=small.index)

    n_random=n_total-len(small)
    if n_random>0 and len(remain):
        choose=rng.choice(
            remain.index.to_numpy(),
            size=min(n_random,len(remain)),
            replace=False
        )
        sample=pd.concat([small,remain.loc[choose]])
    else:
        sample=small

    ims=[]
    for _,r in sample.iterrows():
        orig={"cx":r.cx,"cy":r.cy,"w":r.w,"h":r.h}
        new={"cx":r.new_cx,"cy":r.new_cy,"w":r.new_w,"h":r.new_h}
        ims.append(draw_pair(r.image_path,orig,new,cls))

    cols=4
    rows=math.ceil(len(ims)/cols)
    sheet=Image.new("RGB",(cols*420,rows*320))

    for i,im in enumerate(ims):
        sheet.paste(im,((i%cols)*420,(i//cols)*320))

    p=PREVIEW/f"{cls}_original_red_roiA_green.jpg"
    sheet.save(p,quality=90)
    preview_paths.append(p)

print("QA sheets:")
for p in preview_paths:
    print(" ",p)


## 7. Build ROI-A TRAIN/VAL runtime

The image split is unchanged. Only the three locked defect-class bbox policies are transformed.
The number of images and boxes must remain identical to V6.2E TRAIN/VAL.


In [ ]:
if RUNTIME.exists():
    shutil.rmtree(RUNTIME)

for sp in ["train","val"]:
    (RUNTIME/"images"/sp).mkdir(parents=True,exist_ok=True)
    (RUNTIME/"labels"/sp).mkdir(parents=True,exist_ok=True)

# Index transformed boxes by manifest row_id + source label line.
tb_index={}
for _,r in tboxes.iterrows():
    tb_index[(int(r.row_id),int(r.line_no))]=r

runtime_records=[]

for _,imr in images_df.sort_values("row_id").iterrows():
    sp=imr["split"]
    if sp not in {"train","val"}:
        continue

    row_id=int(imr.row_id)
    src_img=Path(imr.image_path)
    ext=src_img.suffix.lower()
    name=f"{imr.sha256[:16]}_{row_id:06d}{ext}"

    dst_img=RUNTIME/"images"/sp/name
    dst_lab=RUNTIME/"labels"/sp/Path(name).with_suffix(".txt")

    try:
        os.link(src_img,dst_img)
    except Exception:
        os.symlink(src_img,dst_img)

    orig=parse_label(imr.label_path)

    with open(dst_lab,"w") as f:
        for b in orig:
            tr=tb_index[(row_id,int(b["line_no"]))]
            f.write(
                f"{int(tr.class_id)} "
                f"{float(tr.new_cx):.8f} {float(tr.new_cy):.8f} "
                f"{float(tr.new_w):.8f} {float(tr.new_h):.8f}\n"
            )

    runtime_records.append({
        "row_id":row_id,
        "sha256":imr.sha256,
        "split":sp,
        "image_name":name,
        "source_image_path":imr.image_path,
        "source_label_path":imr.label_path,
    })

(RUNTIME/"data.yaml").write_text(yaml.safe_dump({
    "path":str(RUNTIME),
    "train":"images/train",
    "val":"images/val",
    "names":NAMES,
},sort_keys=False))

runtime_manifest=pd.DataFrame(runtime_records)
runtime_manifest.to_csv(REPORT/"runtime_manifest_train_val.csv",index=False)

print((RUNTIME/"data.yaml").read_text())
print({
    sp:len(list((RUNTIME/"images"/sp).glob("*")))
    for sp in ["train","val"]
})


## 8. Candidate integrity gate

In [ ]:
def parse_runtime_label(path):
    out=[]
    for ln,line in enumerate(Path(path).read_text(errors="ignore").splitlines(),1):
        if not line.strip():
            continue
        p=line.split()
        if len(p)<5:
            raise RuntimeError(f"Malformed {path}:{ln}")
        cid=int(float(p[0]))
        x,y,w,h=map(float,p[1:5])

        if not (0<=cid<len(NAMES)):
            raise RuntimeError(f"Bad class {cid}: {path}:{ln}")
        if not (0<=x<=1 and 0<=y<=1 and 0<w<=1 and 0<h<=1):
            raise RuntimeError(f"Bad bbox {x,y,w,h}: {path}:{ln}")
        if x-w/2 < -1e-6 or y-h/2 < -1e-6:
            raise RuntimeError(f"Bbox exits top/left: {path}:{ln}")
        if x+w/2 > 1+1e-6 or y+h/2 > 1+1e-6:
            raise RuntimeError(f"Bbox exits bottom/right: {path}:{ln}")

        out.append((cid,x,y,w,h))
    return out

candidate_counts={}
candidate_images={}

for sp in ["train","val"]:
    cc=[0]*len(NAMES)
    ci=[0]*len(NAMES)

    labels=sorted((RUNTIME/"labels"/sp).glob("*.txt"))

    for lp in labels:
        boxes=parse_runtime_label(lp)
        seen=set()
        for b in boxes:
            cc[b[0]]+=1
            seen.add(b[0])
        for cid in seen:
            ci[cid]+=1

    candidate_counts[sp]=cc
    candidate_images[sp]=ci

print("Candidate box counts:",candidate_counts)

assert candidate_counts["train"]==EXPECTED_TRAIN_BOXES
assert candidate_counts["val"]==EXPECTED_VAL_BOXES
assert len(list((RUNTIME/"images"/"train").glob("*")))==EXPECTED_SPLITS["train"]
assert len(list((RUNTIME/"images"/"val").glob("*")))==EXPECTED_SPLITS["val"]

qa={
    "candidate_tag":CANDIDATE_TAG,
    "parent_dataset_tag":FINAL_TAG,
    "policy_hash":POLICY_HASH,
    "policy":POLICY_PAYLOAD,
    "split_counts_dev":{
        "train":EXPECTED_SPLITS["train"],
        "val":EXPECTED_SPLITS["val"],
    },
    "box_counts":{
        "train":candidate_counts["train"],
        "val":candidate_counts["val"],
    },
    "test_included":False,
    "test_touched":False,
    "transform_stats":roi_stats.to_dict("records"),
    "geometry_gate":"PASS",
}

(REPORT/"candidate_qa.json").write_text(json.dumps(qa,indent=2))
print(json.dumps(qa,indent=2)[:10000])
print("✅ ROI-A candidate integrity PASS.")


## 9. Persist the candidate label policy before training

Only transformed labels + manifests + QA artifacts are uploaded. The large image files are not
duplicated because they are still the exact V6.2E image bytes and can be reconstructed from the
accepted lineage.


In [ ]:
labels_zip=W/f"{CANDIDATE_TAG}_train_val_labels.zip"

with zipfile.ZipFile(labels_zip,"w",zipfile.ZIP_DEFLATED) as z:
    z.write(RUNTIME/"data.yaml","data.yaml")
    for sp in ["train","val"]:
        for p in (RUNTIME/"labels"/sp).glob("*.txt"):
            z.write(p,Path("labels")/sp/p.name)

stash_put(OUT/"roi_policy.json",f"{REMOTE_DATA}/roi_policy.json")
stash_put(REPORT/"candidate_qa.json",f"{REMOTE_DATA}/candidate_qa.json")
stash_put(REPORT/"roi_transform_stats.csv",f"{REMOTE_DATA}/roi_transform_stats.csv")
stash_put(
    REPORT/"roi_transform_manifest_train_val.csv.gz",
    f"{REMOTE_DATA}/roi_transform_manifest_train_val.csv.gz"
)
stash_put(labels_zip,f"{REMOTE_DATA}/{labels_zip.name}")

for p in PREVIEW.glob("*.jpg"):
    stash_put(p,f"{REMOTE_REPORT}/preview/{p.name}")

print("✅ Candidate policy/labels persisted before training.")


## 10. Exact-control YOLO11n @640 sanity training

Same architecture, resolution, epochs, optimizer, LR schedule, augmentation, and 2×T4 setup as
the accepted V6.2E YOLO11n baseline.

The only intended experimental change is the ROI-A label policy.


In [ ]:
class Uploader(threading.Thread):
    def __init__(self,local,remote,every_min=15):
        super().__init__(daemon=True)
        self.local=Path(local)
        self.remote=remote
        self.every=every_min*60
        self.stop_ev=threading.Event()
        self.last_mtime=None

    def push(self):
        if not self.local.exists():
            return
        m=self.local.stat().st_mtime
        if m==self.last_mtime:
            return

        tmp=self.local.with_suffix(".upload.pt")
        shutil.copy2(self.local,tmp)
        try:
            stash_put(tmp,self.remote)
            self.last_mtime=m
        finally:
            tmp.unlink(missing_ok=True)

    def run(self):
        while not self.stop_ev.wait(self.every):
            self.push()

    def stop(self):
        self.stop_ev.set()
        self.push()

def train_roi_a():
    name="yolo11n_v6p3_roi_a_640"
    run=RUNS/name
    wdir=run/"weights"
    wdir.mkdir(parents=True,exist_ok=True)

    best_l=wdir/"best.pt"
    last_l=wdir/"last.pt"
    remote=f"{REMOTE_CKPT}/{name}"
    done_l=TMP/f"{name}.done.json"

    if stash_cp(f"{remote}/done.json",done_l,required=False):
        stash_cp(f"{remote}/best.pt",best_l,required=True)
        print(f"[{name}] already complete -> reuse best.pt")
        return best_l,json.loads(done_l.read_text())

    resume=last_l.exists() or stash_cp(
        f"{remote}/last.pt",
        last_l,
        required=False
    )

    up=Uploader(
        last_l,
        f"{remote}/last.pt",
        UPLOAD_EVERY_MIN
    )
    up.start()

    t0=time.time()

    try:
        if resume:
            print(f"[{name}] RESUME")
            YOLO(str(last_l)).train(resume=True)
        else:
            print(f"[{name}] FRESH TRAIN")
            YOLO("yolo11n.pt").train(
                data=str(RUNTIME/"data.yaml"),
                project=str(RUNS),
                name=name,
                exist_ok=True,
                device=[0,1],
                seed=SEED,
                workers=WORKERS,
                deterministic=False,

                imgsz=640,
                epochs=50,
                batch=64,

                optimizer="AdamW",
                lr0=1e-3,
                lrf=0.10,
                cos_lr=True,
                warmup_epochs=1.0,
                warmup_bias_lr=0.0,
                weight_decay=5e-4,
                patience=10,

                amp=True,
                cache=False,
                plots=False,

                mosaic=0.25,
                close_mosaic=5,
                mixup=0.0,
                copy_paste=0.0,
                scale=0.15,
                translate=0.05,
                hsv_h=0.01,
                hsv_s=0.35,
                hsv_v=0.25,
                fliplr=0.5,
            )
    finally:
        up.stop()

    if not best_l.exists() and last_l.exists():
        shutil.copy2(last_l,best_l)

    assert best_l.exists()

    info={
        "name":name,
        "base_model":"yolo11n.pt",
        "candidate_tag":CANDIDATE_TAG,
        "parent_dataset_tag":FINAL_TAG,
        "policy_hash":POLICY_HASH,
        "imgsz":640,
        "epochs":50,
        "batch":64,
        "lr0":1e-3,
        "seconds":round(time.time()-t0),
        "test_touched":False,
    }

    csvp=run/"results.csv"
    if csvp.exists():
        rows=list(csv.DictReader(open(csvp)))
        if rows:
            info["last_epoch"]={
                k.strip():v.strip()
                for k,v in rows[-1].items()
            }

    stash_put(best_l,f"{remote}/best.pt")
    if last_l.exists():
        stash_put(last_l,f"{remote}/last.pt")
    if csvp.exists():
        stash_put(csvp,f"{remote}/results.csv")

    done_l.write_text(json.dumps(info,indent=2))
    stash_put(done_l,f"{remote}/done.json")

    return best_l,info

YOLO11N_BEST=None
TRAIN_INFO=None

if TRAIN_YOLO11N:
    YOLO11N_BEST,TRAIN_INFO=train_roi_a()
    print(json.dumps(TRAIN_INFO,indent=2)[:8000])
else:
    print("TRAIN_YOLO11N=False")


## 11. VAL evaluation with Ultralytics 1D/2D AP compatibility

In [ ]:
def evaluate_val(best):
    m=YOLO(str(best)).val(
        data=str(RUNTIME/"data.yaml"),
        split="val",
        imgsz=640,
        batch=32,
        device=0,
        plots=False,
        verbose=False,
    )

    eval_ids=[
        int(x)
        for x in np.asarray(m.box.ap_class_index).tolist()
    ]

    missing=sorted(set(range(len(NAMES)))-set(eval_ids))
    if missing:
        raise RuntimeError(
            "Validation missing evaluated classes: "
            + str([(i,NAMES[i]) for i in missing])
        )

    ap50=np.asarray(m.box.ap50,dtype=float).reshape(-1)
    ap=np.asarray(m.box.ap,dtype=float)

    if ap.ndim==1:
        ap5095=ap
    elif ap.ndim==2:
        ap5095=ap.mean(axis=1)
    else:
        raise RuntimeError(f"Unexpected m.box.ap shape: {ap.shape}")

    if len(ap50)!=len(eval_ids) or len(ap5095)!=len(eval_ids):
        raise RuntimeError(
            f"Metric/class mismatch: ids={len(eval_ids)}, "
            f"ap50={len(ap50)}, ap5095={len(ap5095)}"
        )

    per_class={}
    for pos,cid in enumerate(eval_ids):
        per_class[NAMES[cid]]={
            "mAP50":float(ap50[pos]),
            "mAP50_95":float(ap5095[pos]),
            "gap":float(ap50[pos]-ap5095[pos]),
            "retention":(
                float(ap5095[pos]/ap50[pos])
                if ap50[pos]>0 else None
            ),
        }

    return {
        "precision":float(m.box.mp),
        "recall":float(m.box.mr),
        "mAP50":float(m.box.map50),
        "mAP50_95":float(m.box.map),
        "gap":float(m.box.map50-m.box.map),
        "retention":(
            float(m.box.map/m.box.map50)
            if m.box.map50>0 else None
        ),
        "per_class":per_class,
    }

if YOLO11N_BEST is None:
    raise RuntimeError("No YOLO11n best checkpoint available.")

VAL_RESULT=evaluate_val(YOLO11N_BEST)

print(json.dumps(VAL_RESULT,indent=2))


## 12. Controlled comparison + promotion gate

A pass here means **ROI-A is worth promoting to a full V6.3 freeze candidate**.

It does not mean "the model architecture improved". The annotation target changed on purpose.


In [ ]:
comparison={
    "candidate_tag":CANDIDATE_TAG,
    "parent_dataset_tag":FINAL_TAG,
    "policy_hash":POLICY_HASH,
    "baseline_v6p2e_yolo11n_640":BASELINE_V6P2E_YOLO11N,
    "roi_a_yolo11n_640":VAL_RESULT,
}

comparison["delta"]={
    "mAP50":(
        VAL_RESULT["mAP50"]
        - BASELINE_V6P2E_YOLO11N["mAP50"]
    ),
    "mAP50_95":(
        VAL_RESULT["mAP50_95"]
        - BASELINE_V6P2E_YOLO11N["mAP50_95"]
    ),
    "gap":(
        VAL_RESULT["gap"]
        - BASELINE_V6P2E_YOLO11N["gap"]
    ),
}

gate={
    "required_mAP50":ACCEPT_MAP50,
    "required_max_gap":ACCEPT_MAX_GAP,
    "pass_mAP50":VAL_RESULT["mAP50"]>=ACCEPT_MAP50,
    "pass_gap":VAL_RESULT["gap"]<=ACCEPT_MAX_GAP,
}
gate["promotion_recommended"]=(
    gate["pass_mAP50"] and gate["pass_gap"]
)

comparison["gate"]=gate
comparison["test_touched"]=False

display(pd.DataFrame([
    {
        "dataset":"V6.2E original",
        "mAP50":BASELINE_V6P2E_YOLO11N["mAP50"],
        "mAP50:95":BASELINE_V6P2E_YOLO11N["mAP50_95"],
        "gap":BASELINE_V6P2E_YOLO11N["gap"],
    },
    {
        "dataset":"V6.3-ROI-A candidate",
        "mAP50":VAL_RESULT["mAP50"],
        "mAP50:95":VAL_RESULT["mAP50_95"],
        "gap":VAL_RESULT["gap"],
    },
]))

print(json.dumps(comparison,indent=2)[:12000])

if gate["promotion_recommended"]:
    print("\n✅ PROMOTION RECOMMENDED")
    print("ROI-A passed both development gates.")
    print("Next dataset step: lock this exact policy, mechanically transform TEST labels once,")
    print("freeze V6.3, then run the final held-out evaluation without further tuning.")
else:
    print("\n⛔ DO NOT PROMOTE ROI-A YET")
    print("Do not train RT-DETR on this candidate yet.")
    print("Inspect the per-class result and QA sheets before considering ROI-B (e.g. 1.4×).")

decision_path=REPORT/"roi_a_decision.json"
decision_path.write_text(json.dumps(comparison,indent=2))

stash_put(decision_path,f"{REMOTE_REPORT}/roi_a_decision.json")

print("✅ TEST untouched.")


## 13. Package experiment report

In [ ]:
report_zip=W/f"{CANDIDATE_TAG}_experiment_report.zip"

with zipfile.ZipFile(report_zip,"w",zipfile.ZIP_DEFLATED) as z:
    for p in REPORT.rglob("*"):
        if p.is_file():
            z.write(p,p.relative_to(OUT.parent))
    for p in PREVIEW.rglob("*"):
        if p.is_file():
            z.write(p,p.relative_to(OUT.parent))
    z.write(OUT/"roi_policy.json",(OUT/"roi_policy.json").relative_to(OUT.parent))

stash_put(report_zip,f"{REMOTE_REPORT}/{report_zip.name}")

print("Report:",report_zip)
print("Candidate:",CANDIDATE_TAG)
print("TEST touched:",False)
